# Sesion 03 – Ingesta Incremental con Auto Loader
## Certificacion Databricks Data Engineer Associate

**Runtime minimo:** Databricks 13.3 LTS  
**Plataforma:** Azure Databricks con Unity Catalog habilitado  
**Dataset:** Ordenes de compra – Distribuidora automotriz (3 lotes CSV)

### Pasos del laboratorio
1. Configuracion de rutas y constantes
2. Verificar archivos en el Volume
3. Lote 1: pipeline base con esquema inferido y schema evolution activado
4. Lote 2: nuevo campo `prioridad_envio` absorbido por `addNewColumns`
5. Lote 3: campo totalmente fuera del esquema fijo capturado por `rescuedDataColumn`
6. Historial Delta y estadisticas finales

> **Diferencia clave entre Paso 4 y Paso 5:**  
> Paso 4 usa esquema dinamico (`addNewColumns`): columnas nuevas se agregan automaticamente.  
> Paso 5 usa esquema fijo (`StructType` explicito): campos fuera del esquema van a `_rescued_data`.

## Instrucciones de carga de archivos

Sube los archivos al Volume antes de ejecutar:

```
/Volumes/ct_eusgroup_core_desa/default/vol_landing/ordenes/lote1/ordenes_lote1_20240301.csv
/Volumes/ct_eusgroup_core_desa/default/vol_landing/ordenes/lote2/ordenes_lote2_20240302.csv
/Volumes/ct_eusgroup_core_desa/default/vol_landing/ordenes/lote3/ordenes_lote3_corrupto_20240303.csv
```

En clase: subir lote1 primero. Lote2 y lote3 se agregan en sus pasos correspondientes.

## Paso 1 – Configuracion de rutas y constantes

In [0]:
# =============================================================
# RUTAS ESTANDAR DEL LABORATORIO
# =============================================================

CATALOGO        = "ct_eusgroup_core_desa"
ESQUEMA         = "default"
VOL_LANDING     = f"/Volumes/{CATALOGO}/{ESQUEMA}/vol_landing"

SOURCE_PATH     = f"{VOL_LANDING}/ordenes"
CHECKPOINT_PATH = f"{VOL_LANDING}/checkpoints/ordenes_autoloader"
SCHEMA_PATH     = f"{VOL_LANDING}/schema_store/ordenes_autoloader"

# Checkpoint separado para el pipeline de esquema fijo (Paso 5)
CHECKPOINT_PATH_FIXED = f"{VOL_LANDING}/checkpoints/ordenes_fixed_schema"

TABLA_DESTINO   = f"`{CATALOGO}`.`{ESQUEMA}`.bronze_ordenes"
TABLA_CUARENTENA = f"`{CATALOGO}`.`{ESQUEMA}`.bronze_ordenes_cuarentena"

print(f"SOURCE_PATH          : {SOURCE_PATH}")
print(f"CHECKPOINT_PATH      : {CHECKPOINT_PATH}")
print(f"CHECKPOINT_PATH_FIXED: {CHECKPOINT_PATH_FIXED}")
print(f"SCHEMA_PATH          : {SCHEMA_PATH}")
print(f"TABLA_DESTINO        : {TABLA_DESTINO}")
print()
print("ATENCION: checkpointLocation != schemaLocation. Nunca usar el mismo path para ambos.")

## Paso 2 – Verificar archivos en el Volume

In [0]:
archivos = dbutils.fs.ls(SOURCE_PATH)
for a in archivos:
    print(a.path)

In [0]:
# Preview del lote 1
preview = spark.read.option("header", True).csv(f"{SOURCE_PATH}/lote1/")
print(f"Filas: {preview.count()} | Columnas: {len(preview.columns)}")
print(preview.columns)
preview.show(3, truncate=True)

## Paso 3 – Configurar y ejecutar pipeline: Lote 1

Esquema inferido + `addNewColumns` + `rescuedDataColumn`.  
`_metadata.file_name` es la forma correcta de capturar el nombre del archivo fuente en DBR 13.3+.

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_stream1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", 10)
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(SOURCE_PATH)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_name"))  # correcto en DBR 13.3+
)

print("Schema del stream (lote 1):")
df_stream1.printSchema()

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLA_DESTINO}
    USING DELTA
    COMMENT 'Tabla bronze de ordenes - ingesta con Auto Loader'
""")

query1 = (
    df_stream1.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")  # necesario cuando el esquema puede evolucionar
    .trigger(availableNow=True)
    .toTable(f"{CATALOGO}.{ESQUEMA}.bronze_ordenes")
)
query1.awaitTermination()
print("Lote 1 procesado")

In [0]:
df_v1 = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_ordenes")
print(f"Filas en tabla: {df_v1.count()}")
print(f"Columnas: {df_v1.columns}")
df_v1.show(3, truncate=True)

## Paso 4 – Lote 2: schema evolution con `addNewColumns`

**ACCION REQUERIDA:** Sube `ordenes_lote2_20240302.csv` a `{SOURCE_PATH}/lote2/`

El Lote 2 tiene una columna nueva: `prioridad_envio` (NORMAL / EXPRESS / URGENTE).  
Con `schemaEvolutionMode = addNewColumns`, Auto Loader detecta la columna y la agrega al esquema  
y a la tabla Delta automaticamente. Las filas del Lote 1 quedaran con NULL en esa columna.

In [0]:
# Verificar que lote2 esta disponible
try:
    for a in dbutils.fs.ls(f"{SOURCE_PATH}/lote2/"):
        print(a.path)
except:
    print(f"Sube el archivo a {SOURCE_PATH}/lote2/ antes de continuar")

In [0]:
# Relanzar con el mismo checkpoint: Auto Loader solo procesara lote2
df_stream2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", 10)
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(SOURCE_PATH)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_name"))
)

query2 = (
    df_stream2.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOGO}.{ESQUEMA}.bronze_ordenes")
)
query2.awaitTermination()
print("Lote 2 procesado")

In [0]:
df_v2 = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_ordenes")
print(f"Total filas: {df_v2.count()}")
print(f"Columnas: {df_v2.columns}")
print()
# prioridad_envio debe ser NULL para lote1 y tener valor para lote2
print("Distribucion de prioridad_envio por lote:")
df_v2.groupBy("lote_proceso", "prioridad_envio").count().orderBy("lote_proceso", "prioridad_envio").show(20)
print()
print("Verificacion de no reprocesamiento (lote1 no debe aparecer de nuevo):")
df_v2.groupBy("lote_proceso").count().orderBy("lote_proceso").show()

## Paso 5 – Lote 3: esquema fijo y `rescuedDataColumn`

**ACCION REQUERIDA:** Sube `ordenes_lote3_corrupto_20240303.csv` a `{SOURCE_PATH}/lote3/`

Este paso usa un **esquema explicito fijo** (`StructType`), no inferencia dinamica.  
El Lote 3 contiene un campo adicional: `campo_extra_inesperado`.  
Con esquema fijo, ese campo no puede agregarse al esquema, por lo que Auto Loader  
lo captura en la columna `_rescued_data` como JSON.

> **Por que un checkpoint separado:**  
> Este pipeline tiene una configuracion distinta (esquema fijo vs. inferido).  
> Compartir checkpoint con el pipeline anterior causaria conflictos de estado.  
> En produccion, pipelines con configuraciones distintas siempre tienen checkpoints distintos.

> **Por que una tabla distinta:**  
> La tabla `bronze_ordenes` ya tiene el esquema expandido por lote2.  
> Para demostrar `rescuedDataColumn` limpiamente usamos `bronze_ordenes_cuarentena`  
> con esquema fijo basado en lote1 (sin `prioridad_envio`).

In [0]:
try:
    for a in dbutils.fs.ls(f"{SOURCE_PATH}/lote3/"):
        print(a.path)
except:
    print(f"Sube el archivo a {SOURCE_PATH}/lote3/ antes de continuar")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Esquema fijo basado en lote1 (18 columnas conocidas, sin prioridad_envio)
# Cualquier campo fuera de este esquema ira a _rescued_data
schema_fijo = StructType([
    StructField("orden_id",                StringType(),  True),
    StructField("lote_proceso",            StringType(),  True),
    StructField("fecha_orden",             StringType(),  True),
    StructField("cliente_id",              StringType(),  True),
    StructField("cliente_nombre",          StringType(),  True),
    StructField("ciudad_destino",          StringType(),  True),
    StructField("producto_sku",            StringType(),  True),
    StructField("producto_nombre",         StringType(),  True),
    StructField("categoria",               StringType(),  True),
    StructField("almacen_origen",          StringType(),  True),
    StructField("cantidad",                IntegerType(), True),
    StructField("precio_unitario",         DoubleType(),  True),
    StructField("descuento_pct",           IntegerType(), True),
    StructField("precio_final_unitario",   DoubleType(),  True),
    StructField("total_linea",             DoubleType(),  True),
    StructField("canal_venta",             StringType(),  True),
    StructField("estado_orden",            StringType(),  True),
    StructField("dias_entrega_prometidos", IntegerType(), True),
])

# Con esquema fijo NO se usa schemaLocation ni schemaEvolutionMode
# Los campos fuera del esquema (prioridad_envio, campo_extra_inesperado) van a _rescued_data
df_stream3 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.maxFilesPerTrigger", 10)
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(schema_fijo)
    .load(f"{SOURCE_PATH}/lote3/")  # solo lote3 para este pipeline
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_name"))
)

print("Schema del stream con esquema fijo:")
df_stream3.printSchema()

In [0]:
query3 = (
    df_stream3.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH_FIXED)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOGO}.{ESQUEMA}.bronze_ordenes_cuarentena")
)
query3.awaitTermination()
print("Lote 3 procesado")

In [0]:
df_cuarentena = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_ordenes_cuarentena")
print(f"Total filas en tabla cuarentena: {df_cuarentena.count()}")
print(f"Columnas: {df_cuarentena.columns}")
print()

# _rescued_data debe tener contenido: prioridad_envio y campo_extra_inesperado
# ambos estan fuera del esquema fijo
df_rescatados = df_cuarentena.filter(df_cuarentena["_rescued_data"].isNotNull())
print(f"Filas con _rescued_data no nulo: {df_rescatados.count()}")
print()
print("Contenido de _rescued_data (campos fuera del esquema fijo):")
df_rescatados.select("orden_id", "_rescued_data").show(10, truncate=False)

## Paso 6 – Historial Delta y estadisticas finales

In [0]:
# Historial de escrituras en la tabla principal
spark.sql(f"DESCRIBE HISTORY {TABLA_DESTINO}").select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

In [0]:
print("Resumen final – tabla bronze_ordenes (lotes 1 y 2):")
spark.sql(f"""
    SELECT
        lote_proceso,
        COUNT(*) AS total_ordenes,
        COUNT(DISTINCT cliente_id) AS clientes_unicos,
        ROUND(SUM(total_linea), 2) AS facturacion_total,
        SUM(CASE WHEN prioridad_envio IS NOT NULL THEN 1 ELSE 0 END) AS con_prioridad_envio
    FROM {TABLA_DESTINO}
    GROUP BY lote_proceso
    ORDER BY lote_proceso
""").show(truncate=False)

print("Resumen – tabla bronze_ordenes_cuarentena (lote 3, esquema fijo):")
spark.sql(f"""
    SELECT
        lote_proceso,
        COUNT(*) AS total_registros,
        COUNT(_rescued_data) AS con_rescued_data
    FROM {TABLA_CUARENTENA}
    GROUP BY lote_proceso
""").show(truncate=False)

## Limpieza del entorno (opcional)

Descomentar para reiniciar el laboratorio desde cero.

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_DESTINO}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_CUARENTENA}")
# dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
# dbutils.fs.rm(CHECKPOINT_PATH_FIXED, recurse=True)
# dbutils.fs.rm(SCHEMA_PATH, recurse=True)
# print("Entorno limpiado")

print("Celda de limpieza lista (comentada).")